In [1]:
# ====================================================================
# 🧬 NOTEBOOK : POISSON ULTIMATE MIX (HYBRID THEORY)
# Stratégie : Fusion Nucléaire (Poisson + LogLoss) + Sniper Features
# ====================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

# 1. CHARGEMENT & SUPERIOR FEATURE ENGINEERING
# --------------------------------------------------------------------
print("⏳ Chargement des données...")
train_data = pd.read_csv('conversion_data_train.csv')
test_data = pd.read_csv('conversion_data_test.csv')

def feature_engineering(df):
    df_eng = df.copy()
    # --- FEATURES CRITIQUES ---
    # 1. Le 'Active User' Flag : La distinction la plus nette
    df_eng['is_active'] = (df_eng['total_pages_visited'] > 2).astype(int)
    
    # 2. Densité de navigation (Vitesse)
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    
    # 3. Interaction intense
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    return df_eng

# On applique sur tout le monde
X = feature_engineering(train_data.drop('converted', axis=1))
y = train_data['converted']
X_test = feature_engineering(test_data)

# 2. ENCODAGE UNIFIÉ
# --------------------------------------------------------------------
categorical_cols = ['country', 'source']
cat_indices = [X.columns.get_loc(col) for col in categorical_cols]

for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]])
    le.fit(combined)
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

# 3. MOTEUR HYBRIDE : POISSON + LOGLOSS (10-FOLD)
# --------------------------------------------------------------------
n_folds = 10
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds_accumulated = np.zeros(len(X_test))

print(f"🚀 Démarrage du Moteur Hybride (10 Folds)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
    
    # --- MODELE A : POISSON (Le Spécialiste de l'Intensité) ---
    model_poisson = HistGradientBoostingRegressor(
        loss='poisson',
        learning_rate=0.05, max_iter=500, max_depth=8, l2_regularization=0.1,
        categorical_features=cat_indices, random_state=42+fold
    )
    model_poisson.fit(X_train_fold, y_train_fold)
    
    # Prédictions Poisson (Valeurs continues, parfois > 1, mais corrélées au taux)
    val_pred_poi = model_poisson.predict(X_val_fold)
    test_pred_poi = model_poisson.predict(X_test)
    
    # --- MODELE B : CLASSIFIER (Le Probabiliste Classique) ---
    model_clf = HistGradientBoostingClassifier(
        loss='log_loss',
        learning_rate=0.05, max_iter=500, max_depth=8, l2_regularization=0.1,
        categorical_features=cat_indices, random_state=42+fold
    )
    model_clf.fit(X_train_fold, y_train_fold)
    
    # Prédictions Proba (Bornées [0, 1])
    val_pred_clf = model_clf.predict_proba(X_val_fold)[:, 1]
    test_pred_clf = model_clf.predict_proba(X_test)[:, 1]
    
    # --- FUSION (The Ultimate Mix) ---
    # On fait une moyenne simple. Poisson donne souvent un signal plus fort sur les cas limites.
    val_fold_mix = (val_pred_poi + val_pred_clf) / 2
    test_fold_mix = (test_pred_poi + test_pred_clf) / 2
    
    oof_preds[val_idx] = val_fold_mix
    test_preds_accumulated += test_fold_mix
    
    auc_fold = roc_auc_score(y_val_fold, val_fold_mix)
    print(f"  > Fold {fold+1}/{n_folds} | Mix ROC-AUC: {auc_fold:.5f}")

# 4. OPTIMISATION ET GÉNÉRATION
# --------------------------------------------------------------------
test_preds_final = test_preds_accumulated / n_folds

print("\n🔍 Recherche Seuil Optimal...")
best_f1 = 0
best_threshold = 0
thresholds = np.linspace(oof_preds.min(), oof_preds.max(), 1000)

for t in thresholds:
    preds_binary = (oof_preds >= t).astype(int)
    score = f1_score(y, preds_binary)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print(f"\n🏆 RÉSULTATS ULTIMATE MIX :")
print(f"Meilleur F1 OOF : {best_f1:.5f}")
print(f"Seuil Optimal   : {best_threshold:.6f}")

final_test_binary = (test_preds_final >= best_threshold).astype(int)
submission = pd.DataFrame({'converted': final_test_binary})
submission.to_csv('submission_ULTIMATE_MIX.csv', index=False)

print(f"\n✅ Fichier 'submission_ULTIMATE_MIX.csv' généré avec {final_test_binary.sum()} conversions.")

⏳ Chargement des données...
🚀 Démarrage du Moteur Hybride (10 Folds)...
  > Fold 1/10 | Mix ROC-AUC: 0.98689
  > Fold 2/10 | Mix ROC-AUC: 0.98391
  > Fold 3/10 | Mix ROC-AUC: 0.98366
  > Fold 4/10 | Mix ROC-AUC: 0.98633
  > Fold 5/10 | Mix ROC-AUC: 0.98780
  > Fold 6/10 | Mix ROC-AUC: 0.98452
  > Fold 7/10 | Mix ROC-AUC: 0.98290
  > Fold 8/10 | Mix ROC-AUC: 0.98508
  > Fold 9/10 | Mix ROC-AUC: 0.98799
  > Fold 10/10 | Mix ROC-AUC: 0.98620

🔍 Recherche Seuil Optimal...

🏆 RÉSULTATS ULTIMATE MIX :
Meilleur F1 OOF : 0.76877
Seuil Optimal   : 0.391421

✅ Fichier 'submission_ULTIMATE_MIX.csv' généré avec 912 conversions.
